In [ ]:
import json
from pathlib import Path

INPUT  = "medical_qa_1000.jsonl"
OUTPUT = "medical_qa_1000_clean.jsonl"

records = []
with open(INPUT, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

seen    = set()
cleaned = []
dupes   = []

for i, rec in enumerate(records):
    q = rec.get("question", "")
    if q in seen:
        dupes.append((i, q))
    else:
        seen.add(q)
        cleaned.append(rec)

with open(OUTPUT, "w", encoding="utf-8") as f:
    for rec in cleaned:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

print(f"Original : {len(records)} records")
print(f"Removed  : {len(dupes)} duplicates")
print(f"Clean    : {len(cleaned)} records -> {OUTPUT}")

if dupes:
    print("\nRemoved duplicate indices (kept first occurrence):")
    for idx, q in dupes:
        print(f"  #{idx:04d}  {q[:80]}")

Original : 1000 records
Removed  : 114 duplicates
Clean    : 886 records -> medical_qa_1000_clean.jsonl

Removed duplicate indices (kept first occurrence):
  #0100  Which hormone deficiency is implicated in the Costello syndrome ?
  #0117  Which JAK (Janus kinase) inhibitor is approved for treatment of rheumatoid arthr
  #0122  What tyrosine kinase, involved in a Philadelphia- chromosome positive chronic my
  #0153  Which tumor suppressor is referred to as "the guardian of the genome"?
  #0154  Mutation of which gene is implicated in the familial isolated pituitary adenoma?
  #0174  Which calcium/calmodulin dependent protein phosphatase is involved in the activa
  #0195  Which tumor suppressor is referred to as "the guardian of the genome"?
  #0211  What tyrosine kinase, involved in a Philadelphia- chromosome positive chronic my
  #0229  What tyrosine kinase, involved in a Philadelphia- chromosome positive chronic my
  #0232  Which is the transcript responsible for X-chromosome inactiv

In [7]:
"""
merge_medrag_results.py
───────────────────────
Merges all medrag result JSONL files in range order,
deduplicates by `question` (keeps first occurrence),
and writes medrag_results_all.jsonl.
"""

import json
from pathlib import Path

# Files in intended order
FILES = [
    "medrag_results.jsonl",          # 0-299
    "medrag_results(300-399).jsonl",
    "medrag_results(400-499).jsonl",
    "medrag_results(500-599).jsonl",
    "medrag_results(600-799).jsonl",
    "medrag_results(800-899).jsonl",
    "medrag_results(900-1000).jsonl",
]

OUTPUT = "medrag_results_clean.jsonl"

seen    = set()
merged  = []
dupes   = []

for fname in FILES:
    path = Path(fname)
    if not path.exists():
        print(f"WARNING: {fname} not found, skipping.")
        continue

    file_records = 0
    file_dupes   = 0
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rec = json.loads(line)
            q   = rec.get("question", "")
            file_records += 1
            if q in seen:
                dupes.append((fname, q))
                file_dupes += 1
            else:
                seen.add(q)
                merged.append(rec)

    print(f"  {fname}: {file_records} records, {file_dupes} duplicates removed")

with open(OUTPUT, "w", encoding="utf-8") as f:
    for rec in merged:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

print(f"\nTotal duplicates removed : {len(dupes)}")
print(f"Final record count       : {len(merged)}")
print(f"Output                   : {OUTPUT}")

if dupes:
    print("\nDuplicate questions (removed):")
    for fname, q in dupes:
        print(f"  [{fname}] {q[:80]}")

  medrag_results.jsonl: 280 records, 1 duplicates removed
  medrag_results(300-399).jsonl: 100 records, 9 duplicates removed
  medrag_results(400-499).jsonl: 95 records, 8 duplicates removed
  medrag_results(500-599).jsonl: 100 records, 10 duplicates removed
  medrag_results(600-799).jsonl: 194 records, 27 duplicates removed
  medrag_results(800-899).jsonl: 99 records, 16 duplicates removed
  medrag_results(900-1000).jsonl: 98 records, 9 duplicates removed

Total duplicates removed : 80
Final record count       : 886
Output                   : medrag_results_clean.jsonl

Duplicate questions (removed):
  [medrag_results.jsonl] What is the name of Bruton's tyrosine kinase inhibitor that can be used for trea
  [medrag_results(300-399).jsonl] Which gene harbors the mutation T790M?
  [medrag_results(300-399).jsonl] Which is the transcript responsible for X-chromosome inactivation?
  [medrag_results(300-399).jsonl] Which syndrome is NHE6 associated with?
  [medrag_results(300-399).jsonl] Whi